# Regularization in Regression: Ridge, Lasso and Elastic Net

**Experiment:** Predict a continuous outcome using a reproducible synthetic dataset with correlated predictors and irrelevant features. Compare OLS, Ridge, Lasso and Elastic Net using cross-validation, then examine coefficient shrinkage and stability.

All reported results are generated by running the cells; no metrics are manually edited.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
X_arr, y = make_regression(n_samples=900, n_features=20, n_informative=6,
                           n_redundant=5, noise=18, random_state=RANDOM_STATE)
feature_names = [f"feature_{i}" for i in range(X_arr.shape[1])]
X = pd.DataFrame(X_arr, columns=feature_names)
X.head()

## 1. Train-test split and baseline

Scaling is fitted only within training folds through pipelines, preventing leakage. The final test set is held out until model selection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

baseline = Pipeline([("scale", StandardScaler()), ("model", LinearRegression())])
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
print(f"OLS test RMSE: {mean_squared_error(y_test, base_pred)**0.5:.3f}")
print(f"OLS test R²: {r2_score(y_test, base_pred):.3f}")

## 2. Cross-validated regularization search

Use 5-fold CV and a logarithmic penalty grid. Elastic Net's mixing parameter (`l1_ratio`) controls the blend between L1 and L2 penalties.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
alphas = np.logspace(-3, 3, 40)

ridge_pipe = Pipeline([("scale", StandardScaler()), ("model", Ridge())])
ridge_search = GridSearchCV(ridge_pipe, {"model__alpha": alphas}, cv=cv,
                            scoring="neg_root_mean_squared_error", n_jobs=-1)
ridge_search.fit(X_train, y_train)

lasso_pipe = Pipeline([("scale", StandardScaler()), ("model", Lasso(max_iter=20000))])
lasso_search = GridSearchCV(lasso_pipe, {"model__alpha": alphas}, cv=cv,
                            scoring="neg_root_mean_squared_error", n_jobs=-1)
lasso_search.fit(X_train, y_train)

elastic_pipe = Pipeline([("scale", StandardScaler()), ("model", ElasticNet(max_iter=20000))])
elastic_search = GridSearchCV(elastic_pipe,
    {"model__alpha": alphas, "model__l1_ratio": [0.15, 0.35, 0.5, 0.7, 0.85]},
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
elastic_search.fit(X_train, y_train)

for name, search in [("Ridge", ridge_search), ("Lasso", lasso_search), ("Elastic Net", elastic_search)]:
    print(name, "best parameters:", search.best_params_,
          "CV RMSE:", -search.best_score_)

## 3. Test-set comparison

Evaluate the selected models once on the held-out test set.

In [ ]:
models = {"OLS": baseline, "Ridge": ridge_search.best_estimator_,
          "Lasso": lasso_search.best_estimator_,
          "Elastic Net": elastic_search.best_estimator_}
rows = []
for name, model in models.items():
    pred = model.predict(X_test)
    rows.append({"Model": name, "Test RMSE": mean_squared_error(y_test, pred)**0.5,
                 "Test R²": r2_score(y_test, pred)})
results = pd.DataFrame(rows).sort_values("Test RMSE")
results

In [ ]:
results.set_index("Model")["Test RMSE"].plot(kind="bar", title="Held-out Test RMSE", ylabel="RMSE")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 4. Coefficient analysis

Coefficients are compared on standardized features, so their magnitudes are on a common scale. Lasso and Elastic Net may set some coefficients exactly to zero.

In [ ]:
coef_df = pd.DataFrame(index=feature_names)
for name, model in models.items():
    coef_df[name] = model.named_steps["model"].coef_
coef_df.head()

In [ ]:
coef_df.plot(kind="bar", figsize=(14, 5))
plt.title("Coefficients on standardized predictors")
plt.ylabel("Coefficient")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

nonzero = (coef_df.abs() > 1e-8).sum()
print("Number of nonzero coefficients:\n", nonzero)

## 5. Coefficient stability across CV folds

Refit each regularized model on each training fold using its selected hyperparameters. The fold-to-fold coefficient standard deviation provides a rough stability diagnostic.

In [ ]:
def selected_model_with_params(kind, params):
    if kind == "Ridge": return Ridge(alpha=params["model__alpha"])
    if kind == "Lasso": return Lasso(alpha=params["model__alpha"], max_iter=20000)
    return ElasticNet(alpha=params["model__alpha"], l1_ratio=params["model__l1_ratio"], max_iter=20000)

fold_coefs = {kind: [] for kind in ["Ridge", "Lasso", "Elastic Net"]}
for train_idx, _ in cv.split(X_train):
    X_fold, y_fold = X_train.iloc[train_idx], y_train[train_idx]
    for kind, search in [("Ridge", ridge_search), ("Lasso", lasso_search), ("Elastic Net", elastic_search)]:
        scaler = StandardScaler().fit(X_fold)
        xf = scaler.transform(X_fold)
        model = selected_model_with_params(kind, search.best_params_).fit(xf, y_fold)
        fold_coefs[kind].append(model.coef_)
stability = pd.DataFrame({k: np.std(np.vstack(v), axis=0) for k,v in fold_coefs.items()}, index=feature_names)
stability.mean().sort_values()

## 6. Interpretation guide

- **Ridge (L2):** shrinks coefficients continuously; generally retains all features and can help when predictors are correlated.
- **Lasso (L1):** encourages sparse coefficients; may perform feature selection, but correlated variables can compete.
- **Elastic Net:** combines L1 and L2; often useful when groups of predictors are correlated.
- Use the CV-selected hyperparameters, held-out performance, sparsity, and coefficient stability together rather than choosing a model from one metric alone.

## Conclusion

After executing the notebook, summarize the actual results from the test comparison and coefficient diagnostics. Avoid asserting a winner before seeing the generated outputs.